# AI Detector, Probably - logo extractionRuns the UMAP + mirror-graph visuals directly in this notebook kernel (no CLI, no deployed functions).**Setup (once, in the sidebar):**1. **Files panel** -> attach two existing volumes:   - `weightsandotherstuff` (the checkpoint lives at `pangram_final/pangram_best`)   - `pangram-data` (the retrieval index lives at `ai_mirrors.usearch`)2. **Compute profile**: GPU = `A10G`, memory = 16+ GB, idle timeout = 60 min (the UMAP run takes ~15-20 min)3. Run the cells top to bottom.Outputs are saved to `/mnt/weightsandotherstuff/logo/` (persists on the volume) **and** shown inline so you can grab them directly.

In [ ]:
%uv pip install umap-learn sentence-transformers usearch

In [ ]:
import osfrom pathlib import Path# --- adjust these if your volumes mount elsewhere (check the cell output below) ---WEIGHTS_DIR = Path("/mnt/weightsandotherstuff")DATA_DIR    = Path("/mnt/pangram-data")print("Contents of /mnt:")for p in sorted(Path("/mnt").iterdir()):    print("  ", p)print()for d, name in [(WEIGHTS_DIR, "WEIGHTS_DIR"), (DATA_DIR, "DATA_DIR")]:    ok = d.exists()    print(f"{name} = {d}  ->  {'OK' if ok else 'MISSING (attach the volume in the Files panel)'}")    if ok:        for sub in sorted(d.iterdir())[:8]:            print("      ", sub.name)assert WEIGHTS_DIR.exists(), "attach the weightsandotherstuff volume"assert DATA_DIR.exists(), "attach the pangram-data volume"

In [ ]:
import numpy as npimport pandas as pdimport torchimport matplotlib.pyplot as pltfrom datasets import load_dataset# Theme, matching the repo's other plotsBG = "#0d0d0d"; HUMAN_COLOR = "#4dabf7"; AI_COLOR = "#ffa94d"MAX_LENGTH = 512# Held-out essay benchmark sources (same as scripts/eval_essays.py)BENCHMARK_SOURCES = {    "human": [        {"name": "HC3-Human", "text_field": "human_answers", "is_list_field": True,         "data_files": ["https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/refs%2Fconvert%2Fparquet/all/train/0000.parquet"]},        {"name": "Reddit-Writing", "text_field": "content", "is_list_field": False,         "filter_fn": lambda x: len(x.get("content", "").split()) > 150,         "data_files": [f"https://huggingface.co/datasets/webis/tldr-17/resolve/refs%2Fconvert%2Fparquet/default/partial-train/{i:04d}.parquet" for i in range(10)]},    ],    "ai": [        {"name": "HC3-ChatGPT", "text_field": "chatgpt_answers", "is_list_field": True,         "data_files": ["https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/refs%2Fconvert%2Fparquet/all/train/0000.parquet"]},        {"name": "GPT-Wiki-Intro", "text_field": "generated_intro", "is_list_field": False,         "data_files": ["https://huggingface.co/datasets/aadityaubhat/GPT-wiki-intro/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"]},    ],}def load_source(src, max_samples):    ds = load_dataset("parquet", data_files=src["data_files"], split="train", streaming=True)    texts, field, is_list = [], src["text_field"], src.get("is_list_field", False)    filt = src.get("filter_fn", lambda x: True)    for sample in ds:        if len(texts) >= max_samples:            break        if not filt(sample):            continue        if is_list:            answers = sample.get(field, [])            text = answers[0] if answers else ""        else:            text = sample.get(field, "")        if text and len(text.strip()) >= 100:            texts.append(text.strip())    print(f"   {src['name']}: {len(texts):,} samples")    return textsdef load_model(device):    from transformers import DebertaV2ForSequenceClassification, DebertaV2TokenizerFast    ckpt = WEIGHTS_DIR / "pangram_final" / "pangram_best"    print(f"Loading checkpoint {ckpt} ...")    model = DebertaV2ForSequenceClassification.from_pretrained(str(ckpt), num_labels=2)    tokenizer = DebertaV2TokenizerFast.from_pretrained(str(ckpt))    model.eval().to(device)    return model, tokenizerdef embed_cls(texts, model, tokenizer, device, batch_size=16):    """[CLS] embeddings + P(AI) probs, halving the batch on CUDA OOM."""    embs, probs = [], []    bs = batch_size    i = 0    while i < len(texts):        chunk = texts[i:i+bs]        try:            inputs = tokenizer(chunk, truncation=True, max_length=MAX_LENGTH,                               padding=True, return_tensors="pt")            inputs = {k: v.to(device) for k, v in inputs.items()}            with torch.no_grad():                out = model(**inputs)                cls = out.hidden_states[-1][:, 0, :].float().cpu().numpy()                p = torch.softmax(out.logits, dim=-1)[:, 1].float().cpu().numpy()            embs.append(cls); probs.append(p)            i += len(chunk)        except torch.cuda.OutOfMemoryError:            if bs <= 1: raise            torch.cuda.empty_cache(); bs = max(1, bs // 2)    return np.concatenate(embs), np.concatenate(probs)def style(ax):    ax.set_facecolor(BG); ax.set_xticks([]); ax.set_yticks([])    for s in ax.spines.values(): s.set_visible(False)OUT_DIR = WEIGHTS_DIR / "logo"OUT_DIR.mkdir(parents=True, exist_ok=True)print("helpers ready")

In [ ]:
import umapdevice = "cuda" if torch.cuda.is_available() else "cpu"print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "")model, tokenizer = load_model(device)model.config.output_hidden_states = Truetexts, labels, src_names = [], [], []for key, label in [("human", 0), ("ai", 1)]:    for src in BENCHMARK_SOURCES[key]:        t = load_source(src, 1500)        texts += t; labels += [label] * len(t); src_names += [src["name"]] * len(t)n_h = sum(1 for l in labels if l == 0)print(f"Total: {len(texts):,} ({n_h:,} human, {len(labels)-n_h:,} AI)")assert n_h > 0 and n_h < len(labels), "single-class set"print("Extracting [CLS] embeddings ...")emb, probs = embed_cls(texts, model, tokenizer, device)print("Embeddings:", emb.shape)print("Running UMAP ...")xy = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,               metric="cosine", random_state=42).fit_transform(emb)labels_arr = np.array(labels); human = labels_arr == 0fig, ax = plt.subplots(figsize=(9, 9), facecolor=BG); style(ax)ax.scatter(xy[human,0], xy[human,1], s=6, c=HUMAN_COLOR, alpha=0.65, linewidths=0, label="human")ax.scatter(xy[~human,0], xy[~human,1], s=6, c=AI_COLOR, alpha=0.65, linewidths=0, label="AI")ax.legend(facecolor="#1a1a1a", edgecolor="#333333", labelcolor="white", fontsize=13)fig.savefig(OUT_DIR / "umap_ground_truth.png", dpi=300, bbox_inches="tight", facecolor=BG)plt.show(); plt.close(fig)from matplotlib.colors import LinearSegmentedColormapcmap = LinearSegmentedColormap.from_list("h2a", [HUMAN_COLOR, "#f8f9fa", AI_COLOR])fig, ax = plt.subplots(figsize=(9, 9), facecolor=BG); style(ax)sc = ax.scatter(xy[:,0], xy[:,1], s=6, c=probs, cmap=cmap, vmin=0, vmax=1, linewidths=0)cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)cbar.set_label("P(AI)", color="white"); cbar.ax.tick_params(colors="white")cbar.outline.set_edgecolor("#333333")fig.savefig(OUT_DIR / "umap_confidence.png", dpi=300, bbox_inches="tight", facecolor=BG)plt.show(); plt.close(fig)pd.DataFrame({"source": src_names, "label": labels_arr, "ai_prob": probs,              "umap_x": xy[:,0], "umap_y": xy[:,1]}).to_csv(OUT_DIR / "umap_data.csv", index=False)print("saved:", sorted(p.name for p in OUT_DIR.iterdir()))

In [ ]:
from usearch.index import Indexfrom sentence_transformers import SentenceTransformerINDEX_PATH = DATA_DIR / "ai_mirrors.usearch"AI_DIR = DATA_DIR / "ai_corpus"assert INDEX_PATH.exists(), f"index not at {INDEX_PATH}"print("Loading MiniLM index ...")st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")index = Index(ndim=st.get_sentence_embedding_dimension(), metric="cos", dtype="f16")index.load(str(INDEX_PATH))parquet_files = sorted(str(p) for p in AI_DIR.glob("*.parquet"))print(f"corpus: {len(parquet_files)} parquet files")ds = load_dataset("parquet", data_files=parquet_files, split="train")n_humans, top_k = 60, 3texts, src_names = [], []for src in BENCHMARK_SOURCES["human"]:    t = load_source(src, n_humans)    texts += t; src_names += [src["name"]] * len(t)texts, src_names = texts[:n_humans], src_names[:n_humans]print(f"{len(texts)} human texts")print("Scoring humans with the detector ...")_, probs = embed_cls(texts, model, tokenizer, device)print("Searching AI mirrors ...")embs = st.encode(texts, convert_to_numpy=True, normalize_embeddings=True,                 batch_size=256, show_progress_bar=False)matches = index.search(embs, top_k)keys = np.atleast_2d(np.asarray(matches.keys))dists = np.atleast_2d(np.asarray(matches.distances))if keys.shape[0] == 1 and keys.shape[1] != top_k:    keys, dists = keys.T, dists.Tmirror_id_of_text, mirror_texts, pairs = {}, {}, []for h in range(len(texts)):    for k_i in range(keys.shape[1]):        key = int(keys[h, k_i])        if key == -1: continue        try:            mtext = ds[key]["text"]        except (IndexError, KeyError):            continue        if mtext not in mirror_id_of_text:            mirror_id_of_text[mtext] = len(mirror_id_of_text)            mirror_texts[mirror_id_of_text[mtext]] = mtext        sim = 1.0 - float(dists[h, k_i])        pairs.append((h, mirror_id_of_text[mtext], sim))print(f"{len(pairs)} pairs, {len(mirror_texts)} unique mirrors")assert pairs, "no pairs - index/corpus mismatch?"rng = np.random.default_rng(42)order = np.argsort(probs, kind="stable")human_y = {h: rank + rng.uniform(-0.25, 0.25) for rank, h in enumerate(order)}mirror_y = {m: float(np.mean([human_y[h] for h, mm, _ in pairs if mm == m])) for m in mirror_texts}mirror_ids = sorted(mirror_texts, key=lambda m: mirror_y[m])fig, ax = plt.subplots(figsize=(12, 9), facecolor=BG); style(ax)for h, m, sim in pairs:    ax.plot([0, 1], [human_y[h], mirror_y[m]], color="#888888", lw=0.6,            alpha=0.15 + 0.55 * sim, zorder=1)ax.scatter([0.0]*len(texts), [human_y[h] for h in range(len(texts))],           s=[60 + 140 * float(probs[h]) for h in range(len(texts))],           c=HUMAN_COLOR, alpha=0.9, edgecolors="none", zorder=2)ax.scatter([1.0]*len(mirror_ids), [mirror_y[m] for m in mirror_ids],           s=22, c=AI_COLOR, alpha=0.7, edgecolors="none", zorder=2)fig.savefig(OUT_DIR / "mirror_graph.png", dpi=300, bbox_inches="tight", facecolor=BG)plt.show(); plt.close(fig)pd.DataFrame([{"human_source": src_names[h], "human_text": texts[h],               "human_p_ai": float(probs[h]), "mirror_text": mirror_texts[m],               "similarity": float(sim)} for h, m, sim in pairs]             ).to_csv(OUT_DIR / "mirror_pairs.csv", index=False)print("saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## DoneOutputs are in `/mnt/weightsandotherstuff/logo/`:- `umap_ground_truth.png` - human vs AI clusters- `umap_confidence.png` - colored by the model's P(AI); the middle is the hard-negative zone- `mirror_graph.png` - humans (left) -> their AI mirrors (right); node size = P(AI)- `umap_data.csv`, `mirror_pairs.csv` - the underlying points/pairsGrab them via the **Files panel** (they persist on the volume), or just right-click the inline plots above. Drop them in the vault folder and we can pick the logo mark.